# Qarniya AI — dépistage du kératocône (Google Colab ou local)
Équipe NeuroLens · Hackathon Nexus HealthTech 2026

**Avant de commencer :** menu **Exécution → Modifier le type d'exécution → GPU T4 → Enregistrer**.
Puis exécutez les cellules **dans l'ordre** (Maj + Entrée).

Données : CornOrb (Zenodo, CC BY 4.0) — Lazouni et al., DOI 10.5281/zenodo.17121301

## Étape 1 — Vérifier le GPU

In [6]:
!nvidia-smi
import torch
print("GPU disponible :", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activez le GPU : Exécution > Modifier le type d'exécution > GPU T4"

Wed Nov 18 18:29:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.72                 Driver Version: 596.72         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro RTX 4000              WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   59C    P8              6W /   60W |       0MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Étape 2 — Installer Gradio (PyTorch, pandas, sklearn, OpenCV sont déjà dans Colab)

In [ ]:
%pip install -q gradio

## Étape 3 — Télécharger le dataset CornOrb (~680 Mo, 1 à 2 min)

In [14]:
# Fonctionne sur Colab ET en local (Windows/Linux/Mac) : pas de wget/unzip, que du Python.
import os, sys, zipfile, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/qarniya") if IN_COLAB else Path.cwd()   # en local : dossier du notebook
for d in ["data/cornorb", "src", "outputs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print("Dossier du projet :", ROOT)

DATA = ROOT / "data" / "cornorb"
BASE = "https://zenodo.org/api/records/20542091/files/{}/content"

csv = DATA / "clinical_data_and_labels.csv"
if not csv.exists():
    urllib.request.urlretrieve(BASE.format(csv.name), csv)

img_dir = DATA / "ORBSCAN_Dataset"
if not img_dir.exists() or not any(img_dir.rglob("*.png")):
    zp = DATA / "ORBSCAN_Dataset.zip"
    if not zp.exists():
        print("Téléchargement du dataset (~680 Mo)...")
        urllib.request.urlretrieve(BASE.format(zp.name), zp)
    with zipfile.ZipFile(zp) as z:
        z.extractall(DATA)
    zp.unlink()
else:
    print("Dataset déjà présent, téléchargement ignoré.")

print("Nombre d'images :", sum(1 for _ in img_dir.rglob("*.png")))

Dossier du projet : c:\Users\oujdq\Desktop\Quaraniya
Dataset déjà présent, téléchargement ignoré.
Nombre d'images : 5628


## Étape 4 — Écrire le code du projet
Chaque cellule crée un fichier dans `src/`. Exécutez-les toutes.

In [15]:
%%writefile src/config.py
"""Paramètres communs du projet Qarniya AI."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data" / "cornorb"
IMG_DIR = DATA_DIR / "ORBSCAN_Dataset"
CSV_PATH = DATA_DIR / "clinical_data_and_labels.csv"
OUT_DIR = ROOT / "outputs"
MODEL_PATH = OUT_DIR / "qarniya_model.pt"

# Les 4 cartes d'un œil, toujours dans cet ordre
MAPS = ["Axial", "Anterior", "Posterior", "Pachymetry"]

IMG_SIZE = 256        # les PNG font 540x540, on réduit pour aller vite
BATCH_SIZE = 16
EPOCHS = 25
LR = 3e-4
VAL_FOLD = 1          # le fold 1 du train sert de validation
SEED = 42

Overwriting src/config.py


In [16]:
%%writefile src/dataset.py
"""Chargement des données CornOrb : 4 cartes PNG par œil + étiquette (0 normal, 1 kératocône)."""
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import transforms as T

from config import CSV_PATH, IMG_DIR, IMG_SIZE, MAPS, VAL_FOLD

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # normalisation ImageNet


def map_path(patient, eye, m, img_dir=IMG_DIR):
    return img_dir / patient / eye / f"{patient}_{eye}_{m}.png"


def load_table():
    """Lit le CSV et garde seulement les yeux qui ont bien leurs 4 images."""
    df = pd.read_csv(CSV_PATH, dtype={"patient_code": str})
    has_imgs = df.apply(
        lambda r: all(map_path(r.patient_code, r.eye, m).exists() for m in MAPS), axis=1
    )
    missing = (~has_imgs).sum()
    if missing:
        print(f"[info] {missing} yeux sans images ignorés")
    # 46 yeux apparaissent 2 fois dans le CSV avec des valeurs différentes mais un seul
    # jeu d'images : impossible de savoir quelle ligne correspond -> on les exclut.
    dup = df.duplicated(["patient_code", "eye"], keep=False)
    if dup.any():
        print(f"[info] {dup.sum()} lignes en double (même patient + œil) ignorées")
    return df[has_imgs & ~dup].reset_index(drop=True)


def get_splits():
    """Utilise le découpage officiel du dataset (par patient, pas de fuite entre ensembles)."""
    df = load_table()
    test = df[df.split_80_20 == "test"]
    train_all = df[df.split_80_20 == "train"]
    val = train_all[train_all.fold == VAL_FOLD]
    train = train_all[train_all.fold != VAL_FOLD]
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def load_eye(img_paths, eye):
    """Ouvre les 4 cartes. Les yeux gauches (OS) sont retournés en miroir
    pour que nasal/temporal soient du même côté que pour l'œil droit (OD)."""
    imgs = []
    for p in img_paths:
        im = Image.open(p).convert("RGB")
        if eye == "OS":
            im = ImageOps.mirror(im)
        imgs.append(im)
    return imgs


def eval_transform():
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])


def train_transform():
    # Pas de changement de couleur : la couleur EST l'information (échelle de l'appareil)
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.95, 1.05)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])


class CornOrbDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df
        self.tf = train_transform() if train else eval_transform()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        paths = [map_path(r.patient_code, r.eye, m) for m in MAPS]
        imgs = load_eye(paths, r.eye)
        x = torch.stack([self.tf(im) for im in imgs])  # (4, 3, H, W)
        y = torch.tensor(float(r.label))
        return x, y

Overwriting src/dataset.py


In [18]:
%%writefile src/model.py
"""Modèle : un même ResNet18 (pré-entraîné ImageNet) lit chacune des 4 cartes,
puis on fusionne les 4 descripteurs pour donner un score de risque."""
import torch
import torch.nn as nn
from torchvision import models


class QarniyaNet(nn.Module):
    def __init__(self, n_maps=4, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        # on garde tout sauf la dernière couche (fc)
        self.features = nn.Sequential(*list(backbone.children())[:-2])  # -> (B, 512, h, w)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.n_maps = n_maps
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(512 * n_maps, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        # x : (B, 4, 3, H, W)
        b, m, c, h, w = x.shape
        f = self.features(x.view(b * m, c, h, w))       # (B*4, 512, h', w')
        f = self.pool(f).view(b, m * 512)               # (B, 2048)
        return self.head(f).squeeze(1)                  # logits (B,)

Overwriting src/model.py


In [19]:
%%writefile src/metrics.py
"""Calcul des métriques médicales."""
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "accuracy": (tp + tn) / len(y_true),
        "sensibilite": tp / (tp + fn) if tp + fn else 0.0,   # kératocônes bien détectés
        "specificite": tn / (tn + fp) if tn + fp else 0.0,   # normaux bien reconnus
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }


def print_metrics(name, m):
    print(f"--- {name} ---")
    print(f"AUC          : {m['auc']:.3f}")
    print(f"Accuracy     : {m['accuracy']:.3f}")
    print(f"Sensibilité  : {m['sensibilite']:.3f}  (KC détectés)")
    print(f"Spécificité  : {m['specificite']:.3f}  (normaux reconnus)")
    print(f"Matrice      : TN={m['tn']} FP={m['fp']} | FN={m['fn']} TP={m['tp']}")

Overwriting src/metrics.py


In [20]:
%%writefile src/train.py
"""Entraînement du modèle.  Lancer :  python src/train.py   (option : --epochs 3 pour un test rapide)"""
import argparse
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

import config as C
from dataset import CornOrbDataset, get_splits
from metrics import compute_metrics, print_metrics
from model import QarniyaNet


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            p = torch.sigmoid(model(x.to(device)))
        probs += p.float().cpu().tolist(); ys += y.tolist()
    return np.array(ys), np.array(probs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=C.EPOCHS)
    ap.add_argument("--limit", type=int, default=0, help="nb d'yeux max (test rapide)")
    args = ap.parse_args()

    set_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Appareil :", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    train_df, val_df, _ = get_splits()
    if args.limit:
        train_df = train_df.sample(min(args.limit, len(train_df)), random_state=C.SEED)
        val_df = val_df.sample(min(args.limit // 2, len(val_df)), random_state=C.SEED)
    print(f"Train : {len(train_df)} yeux  |  Validation : {len(val_df)} yeux")

    train_dl = DataLoader(CornOrbDataset(train_df, train=True), batch_size=C.BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
    val_dl = DataLoader(CornOrbDataset(val_df), batch_size=C.BATCH_SIZE, num_workers=2)

    model = QarniyaNet().to(device)
    # poids de classe : il y a moins de kératocônes que de normaux
    pos_weight = torch.tensor((train_df.label == 0).sum() / (train_df.label == 1).sum(), device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=C.LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

    C.OUT_DIR.mkdir(exist_ok=True)
    best_auc = -1
    for epoch in range(1, args.epochs + 1):
        model.train()
        total = 0.0
        for x, y in tqdm(train_dl, desc=f"Epoch {epoch}/{args.epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item() * len(y)
        sched.step()

        ys, ps = predict(model, val_dl, device)
        m = compute_metrics(ys, ps)
        print(f"Epoch {epoch:2d}  loss={total / len(train_df):.4f}  val_AUC={m['auc']:.3f}  "
              f"sens={m['sensibilite']:.3f}  spec={m['specificite']:.3f}")
        if m["auc"] > best_auc:
            best_auc = m["auc"]
            torch.save(model.state_dict(), C.MODEL_PATH)
            print(f"   -> meilleur modèle sauvegardé ({C.MODEL_PATH.name})")

    print(f"\nTerminé. Meilleure AUC validation : {best_auc:.3f}")
    print("Étape suivante : python src/evaluate.py")


if __name__ == "__main__":
    main()

Overwriting src/train.py


In [21]:
%%writefile src/evaluate.py
"""Évalue le modèle sauvegardé sur le jeu de TEST (jamais vu pendant l'entraînement).
Lancer :  python src/evaluate.py"""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader

import config as C
from dataset import CornOrbDataset, get_splits
from metrics import compute_metrics, print_metrics
from model import QarniyaNet
from train import predict


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, _, test_df = get_splits()
    model = QarniyaNet(pretrained=False).to(device)
    model.load_state_dict(torch.load(C.MODEL_PATH, map_location=device))

    dl = DataLoader(CornOrbDataset(test_df), batch_size=C.BATCH_SIZE, num_workers=2)
    ys, ps = predict(model, dl, device)
    m = compute_metrics(ys, ps)
    print(f"Test : {len(test_df)} yeux")
    print_metrics("TEST", m)

    # sauvegarde des résultats + courbe ROC (utile pour les slides)
    (C.OUT_DIR / "test_metrics.json").write_text(json.dumps(m, indent=2))
    test_df.assign(prob_kc=ps).to_csv(C.OUT_DIR / "test_predictions.csv", index=False)
    fpr, tpr, _ = roc_curve(ys, ps)
    plt.figure(figsize=(4, 4))
    plt.plot(fpr, tpr, label=f"AUC = {m['auc']:.3f}")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("1 - Spécificité"); plt.ylabel("Sensibilité"); plt.title("Courbe ROC (test)")
    plt.legend(); plt.tight_layout(); plt.savefig(C.OUT_DIR / "roc_test.png", dpi=150)
    print(f"\nFichiers écrits dans {C.OUT_DIR}: test_metrics.json, test_predictions.csv, roc_test.png")


if __name__ == "__main__":
    main()

Overwriting src/evaluate.py


In [23]:
%%writefile src/baseline_tabular.py
"""Point de comparaison simple : régression logistique sur les chiffres du CSV
(Kmax, épaisseur minimale, astigmatisme, asphéricité). Lancer : python src/baseline_tabular.py"""
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import config as C
from metrics import compute_metrics, print_metrics

FEATURES = ["kmax_value_D", "pachy_thinnest_um", "pachy_central_um", "astig_value_D",
            "asphericity_anterior", "asphericity_posterior"]

from dataset import load_table

df = load_table().dropna(subset=FEATURES)  # mêmes yeux que le modèle images
train, test = df[df.split_80_20 == "train"], df[df.split_80_20 == "test"]
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(train[FEATURES], train.label)
print_metrics("Baseline tabulaire (TEST)", compute_metrics(test.label, clf.predict_proba(test[FEATURES])[:, 1]))

Overwriting src/baseline_tabular.py


In [24]:
%%writefile src/predict.py
"""Prédiction pour UN œil + carte de chaleur Grad-CAM sur chacune des 4 cartes.

Exemple :
  python src/predict.py --folder data/cornorb/ORBSCAN_Dataset/9C6SM/OD
  -> écrit outputs/9C6SM_OD_gradcam.png
"""
import argparse
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import ImageOps

import config as C
from dataset import eval_transform, load_eye
from model import QarniyaNet

_model = None


def get_model(device):
    global _model
    if _model is None:
        _model = QarniyaNet(pretrained=False).to(device)
        _model.load_state_dict(torch.load(C.MODEL_PATH, map_location=device))
        _model.eval()
    return _model


def risk_level(p):
    if p < 0.3:
        return "Faible"
    if p < 0.7:
        return "À surveiller"
    return "Élevé"


def analyse_eye(img_paths, eye="OD"):
    """img_paths : 4 chemins dans l'ordre C.MAPS. Retourne (probabilité, [4 overlays RGB])."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model(device)
    imgs = load_eye(img_paths, eye)
    tf = eval_transform()
    x = torch.stack([tf(im) for im in imgs]).unsqueeze(0).to(device)  # (1,4,3,H,W)

    # --- Grad-CAM : gradients de la sortie par rapport à la dernière couche conv ---
    feats = {}
    h = model.features.register_forward_hook(lambda m, i, o: feats.setdefault("a", o))
    logit = model(x)
    h.remove()
    act = feats["a"]                                   # (4, 512, h, w)
    grad = torch.autograd.grad(logit.sum(), act)[0]    # (4, 512, h, w)
    weights = grad.mean(dim=(2, 3), keepdim=True)
    cams = torch.relu((weights * act).sum(1)).detach().cpu().numpy()  # (4, h, w)
    cams /= cams.max() + 1e-8                          # même échelle pour les 4 cartes

    prob = torch.sigmoid(logit).item()
    overlays = []
    for im, cam in zip(imgs, cams):
        if eye == "OS":                                # remettre l'image dans son sens d'origine
            im, cam = ImageOps.mirror(im), cam[:, ::-1]
        # fond en niveaux de gris pour que la chaleur ne se mélange pas aux couleurs de l'appareil
        base = np.array(im.convert("L").convert("RGB")).astype(np.float32)
        heat = cv2.resize(np.ascontiguousarray(cam), (base.shape[1], base.shape[0]))
        color = cv2.applyColorMap(np.uint8(255 * heat), cv2.COLORMAP_JET)[:, :, ::-1]
        alpha = (0.65 * heat)[..., None]               # transparent là où le modèle ne regarde pas
        overlays.append((base * (1 - alpha) + color * alpha).astype(np.uint8))
    return prob, overlays


def paths_from_folder(folder):
    folder = Path(folder)
    out = []
    for m in C.MAPS:
        found = list(folder.glob(f"*{m}*.png")) + list(folder.glob(f"*{m}*.jpg"))
        if not found:
            raise FileNotFoundError(f"Carte '{m}' introuvable dans {folder}")
        out.append(found[0])
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--folder", required=True, help="dossier contenant les 4 cartes d'un œil")
    ap.add_argument("--eye", default=None, help="OD ou OS (déduit du nom du dossier sinon)")
    args = ap.parse_args()

    folder = Path(args.folder)
    eye = args.eye or ("OS" if folder.name.upper() == "OS" else "OD")
    prob, overlays = analyse_eye(paths_from_folder(folder), eye)

    print(f"Score de risque kératocône : {prob:.1%}  ->  {risk_level(prob)}")
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
    for ax, ov, name in zip(axes, overlays, C.MAPS):
        ax.imshow(ov); ax.set_title(name); ax.axis("off")
    fig.suptitle(f"{folder.parent.name} {eye} — risque {prob:.1%} ({risk_level(prob)})")
    C.OUT_DIR.mkdir(exist_ok=True)
    out = C.OUT_DIR / f"{folder.parent.name}_{eye}_gradcam.png"
    fig.tight_layout(); fig.savefig(out, dpi=110)
    print("Image écrite :", out)


if __name__ == "__main__":
    main()

Overwriting src/predict.py


In [25]:
%%writefile src/app.py
"""Interface de démonstration pour le hackathon.  Lancer :  python src/app.py
puis ouvrir http://127.0.0.1:7860 dans le navigateur."""
import gradio as gr

import config as C
from predict import analyse_eye, risk_level

DISCLAIMER = ("⚠️ Outil d'aide au dépistage. Il ne pose pas de diagnostic : "
              "la décision reste celle de l'ophtalmologue.")


def run_single(eye, axial, anterior, posterior, pachy):
    if not all([axial, anterior, posterior, pachy]):
        raise gr.Error("Merci de fournir les 4 cartes.")
    prob, overlays = analyse_eye([axial, anterior, posterior, pachy], eye)
    txt = f"## Score de risque : {prob:.1%} — {risk_level(prob)}\n\n{DISCLAIMER}"
    return txt, [(o, m) for o, m in zip(overlays, C.MAPS)]


def run_compare(eye, a1, b1, c1, d1, a2, b2, c2, d2):
    if not all([a1, b1, c1, d1, a2, b2, c2, d2]):
        raise gr.Error("Merci de fournir les 4 cartes pour les DEUX examens.")
    p1, _ = analyse_eye([a1, b1, c1, d1], eye)
    p2, ov2 = analyse_eye([a2, b2, c2, d2], eye)
    delta = p2 - p1
    if delta > 0.15:
        verdict = "📈 Le score augmente nettement : possible progression, à contrôler."
    elif delta < -0.15:
        verdict = "📉 Le score diminue."
    else:
        verdict = "➖ Score stable entre les deux examens."
    txt = (f"**Examen 1 :** {p1:.1%}  |  **Examen 2 :** {p2:.1%}  |  **Δ = {delta:+.1%}**\n\n"
           f"### {verdict}\n\n{DISCLAIMER}")
    return txt, [(o, m) for o, m in zip(ov2, C.MAPS)]


def map_inputs(suffix=""):
    return [gr.Image(type="filepath", label=f"{m}{suffix}") for m in C.MAPS]


with gr.Blocks(title="Qarniya AI") as demo:
    gr.Markdown("# Qarniya AI — dépistage précoce du kératocône\nÉquipe NeuroLens · Hackathon Nexus HealthTech 2026")
    with gr.Tab("Analyse d'un œil"):
        eye = gr.Radio(["OD", "OS"], value="OD", label="Œil")
        with gr.Row():
            imgs = map_inputs()
        btn = gr.Button("Analyser", variant="primary")
        out_txt = gr.Markdown()
        out_gal = gr.Gallery(label="Zones suspectes (Grad-CAM)", columns=4, height=360)
        btn.click(run_single, [eye] + imgs, [out_txt, out_gal])

    with gr.Tab("Suivi de progression"):
        eye2 = gr.Radio(["OD", "OS"], value="OD", label="Œil")
        gr.Markdown("**Examen 1 (ancien)**")
        with gr.Row():
            e1 = map_inputs(" (1)")
        gr.Markdown("**Examen 2 (récent)**")
        with gr.Row():
            e2 = map_inputs(" (2)")
        btn2 = gr.Button("Comparer", variant="primary")
        out_txt2 = gr.Markdown()
        out_gal2 = gr.Gallery(label="Zones suspectes — examen récent", columns=4, height=360)
        btn2.click(run_compare, [eye2] + e1 + e2, [out_txt2, out_gal2])

if __name__ == "__main__":
    import sys
    # Sur Google Colab, on crée un lien public temporaire (*.gradio.live) pour ouvrir la démo
    in_colab = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in __import__("os").environ
    demo.launch(share=in_colab)

Overwriting src/app.py


## Étape 5 — Point de comparaison : modèle simple sur les chiffres (Kmax, épaisseur…)

In [26]:
import sys
!"{sys.executable}" src/baseline_tabular.py

[info] 2 yeux sans images ignorés
[info] 93 lignes en double (même patient + œil) ignorées
--- Baseline tabulaire (TEST) ---
AUC          : 0.984
Accuracy     : 0.936
Sensibilité  : 0.813  (KC détectés)
Spécificité  : 1.000  (normaux reconnus)
Matrice      : TN=174 FP=0 | FN=17 TP=74


## Étape 6 — Test rapide du pipeline (2 époques, 120 yeux)
Vérifie seulement qu'il n'y a pas d'erreur.

In [27]:
!"{sys.executable}" src/train.py --epochs 2 --limit 120

Appareil : cuda Quadro RTX 4000
[info] 2 yeux sans images ignorés
[info] 93 lignes en double (même patient + œil) ignorées
Train : 120 yeux  |  Validation : 60 yeux
Epoch  1  loss=0.5872  val_AUC=1.000  sens=0.950  spec=1.000
   -> meilleur modèle sauvegardé (qarniya_model.pt)
Epoch  2  loss=0.2114  val_AUC=1.000  sens=1.000  spec=1.000

Terminé. Meilleure AUC validation : 1.000
Étape suivante : python src/evaluate.py



Epoch 1/2: 100%|██████████| 8/8 [00:17<00:00,  1.70it/s]
                                                        

Epoch 2/2:  88%|████████▊ | 7/8 [00:02<00:00,  2.75it/s]
                                                        


## Étape 7 — Entraînement complet (~15–25 min sur T4)
Le meilleur modèle est sauvegardé dans `outputs/qarniya_model.pt`.

In [ ]:
!"{sys.executable}" src/train.py --epochs 15

## Étape 8 — Évaluation sur le jeu de TEST (jamais vu)

In [ ]:
!"{sys.executable}" src/evaluate.py
from IPython.display import Image, display
display(Image("outputs/roc_test.png"))

### Les erreurs du modèle (à montrer au médecin)

In [ ]:
import pandas as pd
p = pd.read_csv("outputs/test_predictions.csv")
p["pred"] = (p.prob_kc >= 0.5).astype(int)
err = p[p.pred != p.label][["patient_code", "eye", "label", "prob_kc", "kmax_value_D", "pachy_thinnest_um"]]
print(len(err), "erreurs sur", len(p))
err

## Étape 9 — Carte de chaleur (Grad-CAM) : un kératocône et un œil normal

In [ ]:
!"{sys.executable}" -W ignore src/predict.py --folder data/cornorb/ORBSCAN_Dataset/PN34D/OS
!"{sys.executable}" -W ignore src/predict.py --folder data/cornorb/ORBSCAN_Dataset/63358/OS
display(Image("outputs/PN34D_OS_gradcam.png"))
display(Image("outputs/63358_OS_gradcam.png"))

### Tester avec VOS images (ex. celles du médecin)
1. Créez un dossier, par ex. `mes_images/P017_OD` (dans le dossier du projet)
2. Déposez-y les 4 cartes ; leurs noms doivent contenir `Axial`, `Anterior`, `Posterior`, `Pachymetry`
3. Exécutez la cellule ci-dessous (mettez `--eye OS` pour un œil gauche)

In [ ]:
!"{sys.executable}" -W ignore src/predict.py --folder mes_images/P017_OD --eye OD
display(Image("outputs/mes_images_OD_gradcam.png"))

## Étape 10 — Démo web
La cellule affiche un lien public **`https://xxxx.gradio.live`** : ouvrez-le (valable 72 h).
Pour arrêter la démo : bouton ■ de la cellule.

In [ ]:
!"{sys.executable}" -W ignore src/app.py

## Étape 11 — Sauvegarder le modèle et les résultats sur votre PC
(Colab efface tout à la fermeture.)

In [ ]:
import shutil
zip_path = shutil.make_archive("qarniya_resultats", "zip", root_dir=".", base_dir="outputs")
shutil.make_archive("qarniya_src", "zip", root_dir=".", base_dir="src")
print("Archives créées :", zip_path, "et qarniya_src.zip")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(zip_path); files.download("qarniya_src.zip")